In [1]:
import warnings
warnings.filterwarnings(action='ignore')
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
import numpy as np


Instructions for updating:
non-resource variables are not supported in the long term


# 다층 퍼셉트론으로 MNIST 손글씨 숫자 데이터를 불러와서 전처리한다.

## MNIST 손글씨 실습은 케라스에서 제공하는 데이터셋을 사용한다.

학습 데이터 60,000개 테스트 데이터 10,000가 제공되며 데이터 이미지 하나가 28 * 28인 픽셀 데이터이고 각 픽셀은 흑백으로 0 ~ 255 사이의 값을 가진다.

<img src="./MNIST1.png" width="300" align="left" />

## keras가 제공하는 MNIST 손글씨 숫자 데이터셋을 불러온다.

In [2]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
print(x_train.shape, y_train.shape, x_test.shape, y_test.shape)

(60000, 28, 28) (60000,) (10000, 28, 28) (10000,)


## 학습 데이터를 검증 작업을 하기 위해서 다시 학습 데이터와 검증 데이터로 분리한다.

6만개의 학습 데이터를 학습 데이터(5만개)와 검증 데이터(1만개)로 분리한다.

학습 중간마다 검증 데이터로 모델의 성능을 측정하면 모델 학습이 제대로 진행되는지 검증 정확도를 알 수 있고 학습 정확도는 올라가는데 검증 정확도가 더 이상 올라가지 않거나 오히려 떨어질 경우 조기 종료를 구현할 수 있다.

In [3]:
x_val = x_train[50000:] # 검증 데이터 피처
x_train = x_train[:50000] # 학습 데이터 피처
y_val = y_train[50000:] # 검증 데이터 피처에 따른 레이블
y_train = y_train[:50000] # 학습 데이터 피처에 따른 레이블
print(x_train.shape, x_val.shape, y_train.shape, y_val.shape)

(50000, 28, 28) (10000, 28, 28) (50000,) (10000,)


학습 데이터를 출력해보면 데이터가 0 ~ 255 사이의 값을 가지는 것을 확인할 수 있다.

In [4]:
print(y_train[0])
for row in x_train[0]:
    for i in row:
        print(f'{i:3d} ', end='')
    print()

5
  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 
  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 
  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 
  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 
  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 
  0   0   0   0   0   0   0   0   0   0   0   0   3  18  18  18 126 136 175  26 166 255 247 127   0   0   0   0 
  0   0   0   0   0   0   0   0  30  36  94 154 170 253 253 253 253 253 225 172 253 242 195  64   0   0   0   0 
  0   0   0   0   0   0   0  49 238 253 253 253 253 253 253 253 253 251  93  82  82  56  39   0   0   0   0   0 
  0   0   0   0   0   0   0  18 219 253 253 253 253 253 198 182 247 241   0   0   0   0   0   

## 다층 퍼셉트론을 사용하는 손글씨 숫자 이미지 분류 작업 흐름도

<img src="./MNIST2.png" width="1200" align="left" />

## 2차원 학습 데이터를 1차원으로 변환한다.

다층 퍼셉트론의 입력값은 무조건 1차원 형태의 배열만 가능하다.  
MNIST 손글씨 숫자 데이터는 2차원이므로 다층 퍼셉트론의 입력값으로 사용할 수 있도록 reshape() 메소드로 1차원 형태로 변환한다.

In [5]:
print(x_train.shape)
# 28행 28열로 구성된 학습 데이터를 784개의 요소를 가지는 1차원 배열로 변환한다.
# np.reshape(x_train, [50000, 784])
# x_train.reshape(50000, 784)
x_train = x_train.reshape(-1, 784) # -1은 행이 개수는 몇개가 나와도 상관없다는 의미이고 784는 열의 개수는 무조건 784개 이다.
print(x_train.shape)
print(x_val.shape)
# 28행 28열로 구성된 검증 데이터를 784개의 요소를 가지는 1차원 배열로 변환한다.
x_val = x_val.reshape(-1, 784)
print(x_val.shape)
print(x_test.shape)
x_test = x_test.reshape(-1, 784)
print(x_test.shape)

(50000, 28, 28)
(50000, 784)
(10000, 28, 28)
(10000, 784)
(10000, 28, 28)
(10000, 784)


## 학습 데이터를 정규화(스케일링)한다.

1차원으로 변경된 데이터를 그대로 다층 퍼셉트론의 입력으로 사용해도 되지만 더 효율적인 학습을 위해서 데이터를 정규화한다. 데이터를 정규화하면 숫자가 작아지므로 모델의 학습 시간을 단축시키고 더 좋은 성능을 보이는 효과가 있다.
MNIST 손글씨 데이터의 모든 값들은 0 ~ 255 범위에 있으므로 255로 나눠서 모든 값들이 0 ~ 1 범위의 값이 되게한다.

MNIST 손글씨 데이터의 각 픽셀의 데이터 타입은 부호 없이 0 ~ 255 범위의 값만 기억하면되므로 데이터 타입이 부호 없는 8비트 정수(uint8)로 되어있다.

In [6]:
print(type(x_train[0][0]))

<class 'numpy.uint8'>


255로 나눠서 0 ~ 1 범위의 실수로 만들어야 하므로 astype() 메소드를 사용해서 실수로 변환한다.

In [7]:
x_train = x_train.astype(np.float32)
print(type(x_train[0][0]))

<class 'numpy.float32'>


In [8]:
x_train /= 255
x_val = x_val.astype(np.float32)
x_val /= 255
x_test = x_test.astype(np.float32)
x_test /= 255

## 레이블 데이터에 One-Hot Encoding을 적용한다.

MNIST 손글씨 숫자 데이터는 0 ~ 9 범위의 숫자로 분류하면 되므로 손실 함수로 크로스 엔트로피를 사용한다.

In [9]:
print(set(y_train))
print(len(set(y_train)))

{np.uint8(0), np.uint8(1), np.uint8(2), np.uint8(3), np.uint8(4), np.uint8(5), np.uint8(6), np.uint8(7), np.uint8(8), np.uint8(9)}
10


크로스 엔트로피를 계산하기 위해서 레이블(y_train, y_val, y_test)을 to_categorical() 메소드로 원-핫 인코딩을 적용한다.  
원-핫 인코딩은 데이터를 수많은 0과 1개의 1로 구별하는 인코딩 방식으로 0으로 이루어진 벡터 집합에서 단 1개의 1의 값으로 해당 데이터를 구별하는 것을 말한다.

to_categorical(원-핫 인코딩을 적용할 데이터, 레이블 개수)

In [10]:
print(y_train[:5])
y_train = tf.keras.utils.to_categorical(y_train, len(set(y_train))) # 학습 데이터에 따른 레이블에 원-핫 인코딩을 적용한다.
for i in y_train[:5]:
    print(i)
y_val = tf.keras.utils.to_categorical(y_val, len(set(y_val))) # 검증 데이터에 따른 레이블에 원-핫 인코딩을 적용한다.
y_test = tf.keras.utils.to_categorical(y_test, len(set(y_test))) # 테스트 데이터에 따른 레이블에 원-핫 인코딩을 적용한다.

[5 0 4 1 9]
[0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]


## 소프트맥스(SoftMax) 작동 원리

argmax() 메소드는 배열에서 가장 큰 값을 찾아서 인덱스를 리턴한다.

In [13]:
a = tf.constant([3, 10, 5])
sess = tf.Session()
print(sess.run(a))
print(sess.run(tf.argmax(a)))

[ 3 10  5]
1


# 전처리된 데이터를 사용하는 다층 퍼셉트론 모델을 만들고 학습시킨다.

## 모델 설명

입력 데이터는 784개의 숫자가 들어있는 1차원 배열 형태이다.  
784개의 입력을 받는 256개의 노드가 1번째 히든 레이어(h1)의 입력으로 사용되고 ReLU 활성화 함수를 거친 1번째 히든 레이어의 출력값을 입력받는 2번째 히든 레이어(h2)에는 128개의 노드가 있다.  
2번째 히든 레이어에는 과적합을 방지하기위해서 10%의 드롭아웃을 적용한다.  
3번째 히든 레이어(logit)는 2번째 히든 레이어의 출력 128를 입력으로 받아 소프트맥스 활성화 함수를 사용하기 위해 10개의 노드가 있다.  
소프트맥스는 분류해야 할 레이블의 총 개수를 k개라고 할 때 k차원의 벡터를 입력받아 각 레이블에 대한 확률을 계산한다.  
예측값(소프트맥스의 출력값)과 실제값의 차이를 계산하기 위해 크로스 엔트로피를 손실 함수로 사용하고 Adam 옵티마이저(최적화 함수)를 사용해서 역전파를 통해 모든 가중치와 바이어스를 최적화 한다.

<img src="./MNIST3.png" width="800" align="left" />

## 다층 퍼셉트론에 사용할 placeholder를 선언한다.

placeholder를 만들 때 2차원 이상의 데이터를 저장해야 할 경우 shape을 이용해서 저장할 데이터의 차원을 지정해야 한다.  
shape의 행 부분을 None으로 지정한 이유는 데이터 개수의 제한없이 입력받기 위해서이다.  
784는 학습 데이터인 28행 28열 이미지를 1차원으로 변경했기 때문이고 10은 레이블에 원-핫 인코딩을 적용했기 때문이다.

In [15]:
X = tf.placeholder(dtype=tf.float32, shape=[None, 784]) # 피처가 저장될 placeholder
Y = tf.placeholder(dtype=tf.float32, shape=[None, 10]) # 레이블이 저장될 placeholder
dropout = tf.placeholder(dtype=tf.float32) # 드롭아웃 비율이 저장될 placeholder

## 다층 퍼셉트론 모델의 히든 레이어 부분을 함수로 만든다.

In [16]:
def mlp(x):
    # 1번째 히든 레이어
    # 2번째 히든 레이어
    # 3번째 히든 레이어
    # 소프트 맥스 활성화 함수에 적용할 값을 리턴시킨다.
    pass

## 오차 함수와 Adam 옵티마이저를 적용해서 역전파를 구현한다.

## 학습시킨다.